Every deployment that publishes a layer adds a version, and old versions pile up. This page shows how to delete them safely: preview first, then delete.

## Setup

This page runs against [moto](https://docs.getmoto.org/), which fakes AWS in memory, so every output below is real and needs no AWS account. In your own code, skip the moto cell and create the client from your own boto3 session.

In [1]:
# Demo only: fake AWS in memory. Skip this cell in your own code.
import moto

mock_aws = moto.mock_aws()
mock_aws.start()

In [2]:
import boto3

boto_ses = boto3.Session(region_name="us-east-1")
lambda_client = boto_ses.client("lambda")

Publish seven versions of one layer:

In [3]:
layer_name = "my_layer"

for i in range(1, 8):
    lambda_client.publish_layer_version(
        LayerName=layer_name,
        Content={"ZipFile": f"version {i}".encode()},
    )

import simple_aws_lambda.api as sal

## The Retention Rules

## Preview First

With the default `real_run=False`, nothing is deleted; you only see what would be:

In [4]:
sal.cleanup_old_layer_versions(
    lambda_client=lambda_client,
    layer_name=layer_name,
    keep_last_n_versions=3,
)

[]

The list is empty: all seven versions were created seconds ago, so the 90 day rule keeps them. To act on version count alone, set the age to 0:

In [5]:
sal.cleanup_old_layer_versions(
    lambda_client=lambda_client,
    layer_name=layer_name,
    keep_last_n_versions=3,
    keep_versions_newer_than_seconds=0,
)

[4, 3, 2, 1]

Still a preview, so all seven versions are there:

In [6]:
layer_versions = sal.list_layer_versions(
    lambda_client=lambda_client,
    layer_name=layer_name,
).all()
[layer_version.version for layer_version in layer_versions]

[7, 6, 5, 4, 3, 2, 1]

## Delete

When the preview looks right, run it again with `real_run=True`:

In [7]:
sal.cleanup_old_layer_versions(
    lambda_client=lambda_client,
    layer_name=layer_name,
    keep_last_n_versions=3,
    keep_versions_newer_than_seconds=0,
    real_run=True,
)

[4, 3, 2, 1]

In [8]:
layer_versions = sal.list_layer_versions(
    lambda_client=lambda_client,
    layer_name=layer_name,
).all()
[layer_version.version for layer_version in layer_versions]

[7, 6, 5]

Running it again deletes nothing, so it is safe to call after every deployment:

In [9]:
sal.cleanup_old_layer_versions(
    lambda_client=lambda_client,
    layer_name=layer_name,
    keep_last_n_versions=3,
    keep_versions_newer_than_seconds=0,
    real_run=True,
)

[]

Stop moto when the demo is done:

In [10]:
mock_aws.stop()